In [8]:
import os
import sys
import numpy as np
import pandas as pd
from ctgan import CTGAN
from sklearn.tree import DecisionTreeClassifier
from art.estimators.classification.scikitlearn import ScikitlearnDecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler

## Trained decision tree model

In [9]:

# Load your dataset and parse dates
data = pd.read_csv('cardio_data.csv', parse_dates=['timestamp'])

# Convert 'timestamp' to a numerical format (e.g., Unix timestamp)
data['timestamp_numeric'] = data['timestamp'].astype('int64') // 1e9  # Convert to seconds since epoch

# Handle categorical columns by encoding them
categorical_columns = ['af_diagnosis', 'af_detected', 'signal_quality', 'af_context_30min']
for col in categorical_columns:
    data[col] = LabelEncoder().fit_transform(data[col])

# Drop the original 'timestamp' column
data = data.drop(columns=['timestamp'])

# Initialize the CTGAN model
ctgan = CTGAN(epochs=1000)

# Fit the CTGAN model to the data
ctgan.fit(data, categorical_columns)

# Generate synthetic data
ddata = ctgan.sample(100000)

# Convert 'timestamp_numeric' back to datetime
ddata['timestamp'] = pd.to_datetime(ddata['timestamp_numeric'], unit='s')

# Drop 'timestamp_numeric' if not needed
ddata = ddata.drop(columns=['timestamp_numeric'])

# Save the synthetic data to a CSV file
ddata.to_csv('ddata.csv', index=False)

# Define feature columns and target column
feature_columns = ['age', 'ecg_reading', 'ppg_reading', 'signal_quality', 'af_context_30min', 'confidence_score']
target_column = 'af_detected'



In [10]:
from sklearn.model_selection import train_test_split

# Split the synthetic data into training and test sets
x_train, x_test, y_train, y_test = train_test_split(
    ddata[feature_columns], ddata[target_column], test_size=0.5, random_state=42
)

# Train Decision Tree Classifier
model = DecisionTreeClassifier()
model.fit(x_train, y_train)
art_classifier = ScikitlearnDecisionTreeClassifier(model)

print('Base model accuracy: ', model.score(x_test, y_test))

Base model accuracy:  0.96506


In [11]:
ddata.to_csv('ddata.csv', index=False)

In [12]:
ddata.tail()

,patient_id,age,af_diagnosis,ecg_reading,ppg_reading,af_detected,signal_quality,af_context_30min,confidence_score,timestamp
99995,3,66,0,0.718430,1.283713,0,0,0,0.920560,2025-01-01 07:59:39.302040815
99996,5,60,1,0.956927,1.635019,1,0,1,0.987262,2025-01-01 08:13:59.885910034
99997,7,60,1,0.818297,1.403895,1,1,1,0.937031,2025-01-01 08:15:00.450591326
99998,6,67,0,0.810270,1.171045,0,2,0,0.880532,2025-01-01 08:13:46.510411024
99999,2,71,0,0.685312,1.056876,0,2,0,0.892564,2025-01-01 08:15:23.545096159
